# Gene → CellularComponent Relation Pipeline

Builds a unified, deduplicated edge table for the **Gene–CellularComponent** relation.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | kg_type | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- Monarch (Human/Gene_Human_CellularComponent_MONARCH.csv - Native Generalised)
- DRKG (DRKG_Gene_Cellular_Component.csv - Native Generalised)
- Hetionet (Gene_CellularComponent_Hetionet.csv - Native Generalised)
- GPKG (Gene_CellularComponent_GPKG.csv - Native Generalised)
- Harmonizome (Gene_CellularComponent_Harmonizome.csv - Native Generalised)

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [1]:
import pandas as pd
import os

BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_CELLULARCOMPONENT/ALL_GENE_CELLULARCOMPONENT.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]

## 1 · Mapping DBs

In [2]:
# MESH mapping to map tail (GO_ID) to name
go_db = pd.read_csv(MAPPING_DIR + 'MESH/MESH_GO_ID_NAME.csv')
GO_dict = dict(zip(go_db['id'], go_db['name']))
del go_db

# NCBI mapping to resolve gene synonyms and descriptions
ncbi = pd.read_csv(MAPPING_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_sym2desc_dict = dict(zip(ncbi['Symbol'],  ncbi['description']))
NCBI_syn2sym_dict = {}
for synonyms, symbol in zip(ncbi['Synonyms'], ncbi['Symbol']):
    for syn in str(synonyms).split('|'):
        NCBI_syn2sym_dict[syn.strip()] = symbol
del ncbi

## 2 · Load Source Files

In [3]:
# 1. Monarch (Gene -> CellularComponent)
Monarch = pd.read_csv(PROC_DIR + 'Monarch/Human/Gene_Human_CellularComponent_MONARCH.csv')
Monarch.columns = Monarch.columns.str.lower()
if 'kgsource' in Monarch.columns:
    Monarch = Monarch.rename(columns={'kgsource': 'kg_source'})
if 'head_name' in Monarch.columns:
    Monarch = Monarch.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in Monarch.columns:
    Monarch = Monarch.rename(columns={'tail_name': 'tail_detail_name'})

Monarch['head_id_is'] = 'NCBI_ID'
Monarch['tail_id_is'] = 'Quick_GO'
Monarch['relation'] = 'Gene_CellularComponent'
Monarch['head_type'] = 'Gene'
Monarch['tail_type'] = 'CellularComponent'
Monarch['kg_source'] = 'Monarch_KG'
Monarch['kg_type'] = 'Generalised'
if 'tail_detail_name' not in Monarch.columns:
    Monarch['tail_detail_name'] = Monarch['tail'].map(GO_dict)
else:
    Monarch['tail_detail_name'] = Monarch['tail_detail_name'].fillna(Monarch['tail'].map(GO_dict))
print(f"Monarch: {len(Monarch):,} rows")

# 2. DRKG (Gene -> CellularComponent)
DRKG = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Gene_Cellular_Component.csv')
DRKG.columns = DRKG.columns.str.lower()
if 'head_name' in DRKG.columns:
    DRKG = DRKG.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in DRKG.columns:
    DRKG = DRKG.rename(columns={'tail_name': 'tail_detail_name'})

DRKG['head_id_is'] = 'NCBI_ID'
DRKG['tail_id_is'] = 'Quick_GO'
DRKG['relation'] = 'Gene_CellularComponent'
DRKG['head_type'] = 'Gene'
DRKG['tail_type'] = 'CellularComponent'
DRKG['kg_source'] = 'DRKG'
DRKG['kg_type'] = 'Generalised'
DRKG['tail_detail_name'] = DRKG['tail'].map(GO_dict)
print(f"DRKG: {len(DRKG):,} rows")

# 3. Hetionet (Gene -> CellularComponent)
Hetionet = pd.read_csv(PROC_DIR + 'Hetionet/Gene_CellularComponent_Hetionet.csv')
Hetionet.columns = Hetionet.columns.str.lower()
if 'head_name' in Hetionet.columns:
    Hetionet = Hetionet.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in Hetionet.columns:
    Hetionet = Hetionet.rename(columns={'tail_name': 'tail_detail_name'})

Hetionet['head_id_is'] = 'NCBI_ID'
Hetionet['tail_id_is'] = 'Quick_GO'
Hetionet['relation'] = 'Gene_CellularComponent'
Hetionet['head_type'] = 'Gene'
Hetionet['tail_type'] = 'CellularComponent'
Hetionet['kg_source'] = 'Hetionet'
Hetionet['kg_type'] = 'Generalised'
if 'tail_detail_name' not in Hetionet.columns:
    Hetionet['tail_detail_name'] = Hetionet['tail'].map(GO_dict)
else:
    Hetionet['tail_detail_name'] = Hetionet['tail_detail_name'].fillna(Hetionet['tail'].map(GO_dict))
print(f"Hetionet: {len(Hetionet):,} rows")

# 4. GPKG (Gene -> CellularComponent)
GPKG = pd.read_csv(PROC_DIR + 'GPKG/Gene_CellularComponent_GPKG.csv')
GPKG.columns = GPKG.columns.str.lower()
if 'head_name' in GPKG.columns:
    GPKG = GPKG.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in GPKG.columns:
    GPKG = GPKG.rename(columns={'tail_name': 'tail_detail_name'})

GPKG['head_id_is'] = 'NCBI_ID'
GPKG['tail_id_is'] = 'Quick_GO'
GPKG['relation'] = 'Gene_CellularComponent'
GPKG['head_type'] = 'Gene'
GPKG['tail_type'] = 'CellularComponent'
GPKG['kg_source'] = 'GPKG'
GPKG['kg_type'] = 'Generalised'
if 'tail_detail_name' not in GPKG.columns:
    GPKG['tail_detail_name'] = GPKG['tail'].map(GO_dict)
else:
    GPKG['tail_detail_name'] = GPKG['tail_detail_name'].fillna(GPKG['tail'].map(GO_dict))
print(f"GPKG: {len(GPKG):,} rows")

# 5. Harmonizome (Gene -> CellularComponent)
harmonizome_path = PROC_DIR + 'harmonizome/Gene_CellularComponent_Harmonizome.csv'
if os.path.exists(harmonizome_path):
    Harmonizome = pd.read_csv(harmonizome_path)
    Harmonizome.columns = Harmonizome.columns.str.lower()
    if 'kgsource' in Harmonizome.columns:
        Harmonizome = Harmonizome.rename(columns={'kgsource': 'kg_source'})
    if 'head_name' in Harmonizome.columns:
        Harmonizome = Harmonizome.rename(columns={'head_name': 'head_detail_name'})
    if 'tail_name' in Harmonizome.columns:
        Harmonizome = Harmonizome.rename(columns={'tail_name': 'tail_detail_name'})
    Harmonizome['head_id_is'] = 'NCBI_ID'
    Harmonizome['tail_id_is'] = 'Quick_GO'
    Harmonizome['relation'] = 'Gene_CellularComponent'
    Harmonizome['head_type'] = 'Gene'
    Harmonizome['tail_type'] = 'CellularComponent'
    Harmonizome['kg_source'] = 'Harmonizome'
    Harmonizome['kg_type'] = 'Generalised'
    if 'tail_detail_name' not in Harmonizome.columns:
        Harmonizome['tail_detail_name'] = Harmonizome['tail'].map(GO_dict)
    else:
        Harmonizome['tail_detail_name'] = Harmonizome['tail_detail_name'].fillna(Harmonizome['tail'].map(GO_dict))
    print(f"Harmonizome: {len(Harmonizome):,} rows")
else:
    print("Warning: Harmonizome path not found")
    Harmonizome = pd.DataFrame(columns=REQUIRED_COLS)

Monarch: 201,613 rows
DRKG: 71,088 rows
Hetionet: 73,564 rows
GPKG: 58,263 rows
Harmonizome: 375,817 rows


In [4]:
source_dfs = {
    "Monarch": Monarch,
    "DRKG": DRKG,
    "Hetionet": Hetionet,
    "GPKG": GPKG,
    "Harmonizome": Harmonizome
}

for name, df in source_dfs.items():
    print(f"Variable name: {name}")
    display(df.head())   # or print(df)


Variable name: Monarch


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_detail_name,tail_detail_name,head_orignal,species_species,kg_type
0,MTHFSD,Gene_CellularComponent,GO:0005737,Gene,is_active_in,CellularComponent,infores:go-central,Monarch_KG,methenyltetrahydrofolate synthetase domain con...,Homo sapiens,NaN,NCBI_ID,Quick_GO,MTHFSD,cytoplasm,HGNC:25778,Homo sapiens_Homo sapiens,Generalised
1,ATRX,Gene_CellularComponent,GO:0005634,Gene,is_active_in,CellularComponent,infores:go-central,Monarch_KG,ATRX chromatin remodeler,Homo sapiens,NaN,NCBI_ID,Quick_GO,ATRX,nucleus,HGNC:886,Homo sapiens_Homo sapiens,Generalised
2,AJUBA,Gene_CellularComponent,GO:0005912,Gene,is_active_in,CellularComponent,infores:go-central,Monarch_KG,ajuba LIM protein,Homo sapiens,NaN,NCBI_ID,Quick_GO,AJUBA,adherens junction,HGNC:20250,Homo sapiens_Homo sapiens,Generalised
3,RPA3,Gene_CellularComponent,GO:0005662,Gene,part_of,CellularComponent,infores:go-central,Monarch_KG,replication protein A3,Homo sapiens,NaN,NCBI_ID,Quick_GO,RPA3,DNA replication factor A complex,HGNC:10291,Homo sapiens_Homo sapiens,Generalised
4,PSMB9,Gene_CellularComponent,GO:0005634,Gene,is_active_in,CellularComponent,infores:go-central,Monarch_KG,proteasome 20S subunit beta 9,Homo sapiens,NaN,NCBI_ID,Quick_GO,PSMB9,nucleus,HGNC:9546,Homo sapiens_Homo sapiens,Generalised


Variable name: DRKG


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_detail_name,head_ens,tail_detail_name,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type
0,KLHL14,Gene_CellularComponent,GO:0043025,Gene,Hetionet::GpCC::Gene:Cellular Component,CellularComponent,DRKG,57565,kelch like family member 14,ENSG00000197705,neuronal cell body,NCBI_ID,Quick_GO,Gene::57565,Cellular Component::GO:0043025,Generalised
1,TTLL5,Gene_CellularComponent,GO:0005874,Gene,Hetionet::GpCC::Gene:Cellular Component,CellularComponent,DRKG,23093,tubulin tyrosine ligase like 5,ENSG00000119685,microtubule,NCBI_ID,Quick_GO,Gene::23093,Cellular Component::GO:0005874,Generalised
2,HSP90AA1,Gene_CellularComponent,GO:0031253,Gene,Hetionet::GpCC::Gene:Cellular Component,CellularComponent,DRKG,3320,heat shock protein 90 alpha family class A mem...,ENSG00000080824,cell projection membrane,NCBI_ID,Quick_GO,Gene::3320,Cellular Component::GO:0031253,Generalised
3,AKAP4,Gene_CellularComponent,GO:0035686,Gene,Hetionet::GpCC::Gene:Cellular Component,CellularComponent,DRKG,8852,A-kinase anchoring protein 4,ENSG00000147081,sperm fibrous sheath,NCBI_ID,Quick_GO,Gene::8852,Cellular Component::GO:0035686,Generalised
4,NUP107,Gene_CellularComponent,GO:0031080,Gene,Hetionet::GpCC::Gene:Cellular Component,CellularComponent,DRKG,57122,nucleoporin 107,ENSG00000111581,nuclear pore outer ring,NCBI_ID,Quick_GO,Gene::57122,Cellular Component::GO:0031080,Generalised


Variable name: Hetionet


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,head_detail_name,tail_detail_name,kg_type
0,CDH1,Gene_CellularComponent,GO:0005912,Gene,Gene–participates–CellularComponent,CellularComponent,Hetionet,NCBI_ID,Quick_GO,999,cadherin 1,adherens junction,Generalised
1,KLHL14,Gene_CellularComponent,GO:0043025,Gene,Gene–participates–CellularComponent,CellularComponent,Hetionet,NCBI_ID,Quick_GO,57565,kelch like family member 14,neuronal cell body,Generalised
2,TTLL5,Gene_CellularComponent,GO:0005874,Gene,Gene–participates–CellularComponent,CellularComponent,Hetionet,NCBI_ID,Quick_GO,23093,tubulin tyrosine ligase like 5,microtubule,Generalised
3,HSP90AA1,Gene_CellularComponent,GO:0031253,Gene,Gene–participates–CellularComponent,CellularComponent,Hetionet,NCBI_ID,Quick_GO,3320,heat shock protein 90 alpha family class A mem...,cell projection membrane,Generalised
4,AKAP4,Gene_CellularComponent,GO:0035686,Gene,Gene–participates–CellularComponent,CellularComponent,Hetionet,NCBI_ID,Quick_GO,8852,A-kinase anchoring protein 4,sperm fibrous sheath,Generalised


Variable name: GPKG


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_orignal,head_alias_mapped,head_detail_name,head_id_is,tail_detail_name,tail_id_is,kg_type
0,NUDT4,Gene_CellularComponent,GO:0005634,Gene,associated,CellularComponent,GPKG,ENSG00000177144,NUDT4B,nudix hydrolase 4,NCBI_ID,nucleus,Quick_GO,Generalised
1,NUDT4,Gene_CellularComponent,GO:0005737,Gene,associated,CellularComponent,GPKG,ENSG00000177144,NUDT4B,nudix hydrolase 4,NCBI_ID,cytoplasm,Quick_GO,Generalised
2,NUDT4,Gene_CellularComponent,GO:0005829,Gene,associated,CellularComponent,GPKG,ENSG00000177144,NUDT4B,nudix hydrolase 4,NCBI_ID,cytosol,Quick_GO,Generalised
3,IGKV2-28,Gene_CellularComponent,GO:0005576,Gene,associated,CellularComponent,GPKG,ENSG00000244116,IGKV2-28,immunoglobulin kappa variable 2-28,NCBI_ID,extracellular region,Quick_GO,Generalised
4,IGKV2-28,Gene_CellularComponent,GO:0005615,Gene,associated,CellularComponent,GPKG,ENSG00000244116,IGKV2-28,immunoglobulin kappa variable 2-28,NCBI_ID,extracellular space,Quick_GO,Generalised


Variable name: Harmonizome


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type
0,PI4K2A,Gene_CellularComponent,GO:0044218,Gene,CellularComponent,COMPARTMENTS Curated Protein Localization Evid...,Harmonizome,phosphatidylinositol 4-kinase type 2 alpha,other organism cell membrane,NCBI_ID,Quick_GO,Generalised
1,PI4K2A,Gene_CellularComponent,GO:0044221,Gene,CellularComponent,COMPARTMENTS Curated Protein Localization Evid...,Harmonizome,phosphatidylinositol 4-kinase type 2 alpha,host cell synapse,NCBI_ID,Quick_GO,Generalised
2,PI4K2A,Gene_CellularComponent,GO:0033644,Gene,CellularComponent,COMPARTMENTS Curated Protein Localization Evid...,Harmonizome,phosphatidylinositol 4-kinase type 2 alpha,host cell membrane,NCBI_ID,Quick_GO,Generalised
3,PI4K2A,Gene_CellularComponent,GO:0044231,Gene,CellularComponent,COMPARTMENTS Curated Protein Localization Evid...,Harmonizome,phosphatidylinositol 4-kinase type 2 alpha,host cell presynaptic membrane,NCBI_ID,Quick_GO,Generalised
4,ANXA13,Gene_CellularComponent,GO:0070382,Gene,CellularComponent,COMPARTMENTS Curated Protein Localization Evid...,Harmonizome,annexin A13,exocytic vesicle,NCBI_ID,Quick_GO,Generalised


## 3 · Consolidate

In [11]:
source_dfs = [Monarch, DRKG, Hetionet, GPKG, Harmonizome]
aligned = []
for df in source_dfs:
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])
final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df[final_df['head'].astype(str).str.upper() != 'NAN']
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,MTHFSD,Gene_CellularComponent,GO:0005737,Gene,is_active_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,methenyltetrahydrofolate synthetase domain con...,cytoplasm
1,ATRX,Gene_CellularComponent,GO:0005634,Gene,is_active_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,ATRX chromatin remodeler,nucleus
2,AJUBA,Gene_CellularComponent,GO:0005912,Gene,is_active_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,ajuba LIM protein,adherens junction
3,RPA3,Gene_CellularComponent,GO:0005662,Gene,part_of,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,replication protein A3,DNA replication factor A complex
4,PSMB9,Gene_CellularComponent,GO:0005634,Gene,is_active_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,proteasome 20S subunit beta 9,nucleus
...,...,...,...,...,...,...,...,...,...,...,...,...
780340,MTFR2,Gene_CellularComponent,GO:0043228,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,mitochondrial fission regulator 2,membraneless organelle
780341,PCBP3,Gene_CellularComponent,GO:0043228,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,poly(rC) binding protein 3,membraneless organelle
780342,RAB11A,Gene_CellularComponent,GO:0043228,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,"RAB11A, member RAS oncogene family",membraneless organelle
780343,RAB11B,Gene_CellularComponent,GO:0043228,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,"RAB11B, member RAS oncogene family",membraneless organelle


In [13]:
final_df[final_df['head'] == 'MED6']

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
578,MED6,Gene_CellularComponent,GO:0016592,Gene,part_of,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,mediator complex
3810,MED6,Gene_CellularComponent,GO:0070847,Gene,part_of,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,core mediator complex
21446,MED6,Gene_CellularComponent,GO:0005654,Gene,located_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,nucleoplasm
21447,MED6,Gene_CellularComponent,GO:0005654,Gene,located_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,nucleoplasm
21448,MED6,Gene_CellularComponent,GO:0005654,Gene,located_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,nucleoplasm
...,...,...,...,...,...,...,...,...,...,...,...,...
604307,MED6,Gene_CellularComponent,GO:0016020,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,membrane
695735,MED6,Gene_CellularComponent,GO:0016592,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,mediator complex
780309,MED6,Gene_CellularComponent,GO:0005856,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,cytoskeleton
780324,MED6,Gene_CellularComponent,GO:0043232,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,mediator complex subunit 6,intracellular membraneless organelle


## 4 · Gene Symbol Normalisation & Pre-dedup NA audit

In [14]:
final_df['head_detail_name'] = final_df['head'].map(NCBI_sym2desc_dict)
final_df

final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)

print(f"Consolidated rows (before deduplication): {len(final_df):,}")
print("\nNaN audit before deduplication:")
print(final_df.isna().sum())

Consolidated rows (before deduplication): 780,139

NaN audit before deduplication:
head                     0
relation                 0
tail                     0
head_type                0
relation_type       375817
tail_type                0
kg_source                0
kg_type                  0
head_id_is               0
tail_id_is               0
head_detail_name         0
tail_detail_name         0
dtype: int64


In [16]:
# final_df[final_df['head'] == 'MED6']
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,MTHFSD,Gene_CellularComponent,GO:0005737,Gene,is_active_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,methenyltetrahydrofolate synthetase domain con...,cytoplasm
1,ATRX,Gene_CellularComponent,GO:0005634,Gene,is_active_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,ATRX chromatin remodeler,nucleus
2,AJUBA,Gene_CellularComponent,GO:0005912,Gene,is_active_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,ajuba LIM protein,adherens junction
3,RPA3,Gene_CellularComponent,GO:0005662,Gene,part_of,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,replication protein A3,DNA replication factor A complex
4,PSMB9,Gene_CellularComponent,GO:0005634,Gene,is_active_in,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,proteasome 20S subunit beta 9,nucleus
...,...,...,...,...,...,...,...,...,...,...,...,...
780134,MTFR2,Gene_CellularComponent,GO:0043228,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,mitochondrial fission regulator 2,membraneless organelle
780135,PCBP3,Gene_CellularComponent,GO:0043228,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,poly(rC) binding protein 3,membraneless organelle
780136,RAB11A,Gene_CellularComponent,GO:0043228,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,"RAB11A, member RAS oncogene family",membraneless organelle
780137,RAB11B,Gene_CellularComponent,GO:0043228,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,"RAB11B, member RAS oncogene family",membraneless organelle


## 5 · Deduplication & Post-dedup NA audit

In [17]:
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']
final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'kg_type': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
print("\nNaN audit after deduplication:")
print(final_df_dedup.isna().sum())

Before dedup: 780,139  |  After dedup: 428,430

NaN audit after deduplication:
head                     0
relation                 0
tail                     0
head_type                0
relation_type       292974
tail_type                0
kg_source                0
kg_type                  0
head_id_is               0
tail_id_is               0
head_detail_name         0
tail_detail_name         0
dtype: int64


In [18]:
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,A1BG,Gene_CellularComponent,GO:0000323,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,alpha-1-B glycoprotein,lytic vacuole
1,A1BG,Gene_CellularComponent,GO:0005575,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,alpha-1-B glycoprotein,cellular_component
2,A1BG,Gene_CellularComponent,GO:0005576,Gene,located_in,CellularComponent,GPKG::Harmonizome::Monarch_KG,Generalised,NCBI_ID,Quick_GO,alpha-1-B glycoprotein,extracellular region
3,A1BG,Gene_CellularComponent,GO:0005615,Gene,located_in,CellularComponent,GPKG::Harmonizome::Monarch_KG,Generalised,NCBI_ID,Quick_GO,alpha-1-B glycoprotein,extracellular space
4,A1BG,Gene_CellularComponent,GO:0005764,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,alpha-1-B glycoprotein,lysosome
...,...,...,...,...,...,...,...,...,...,...,...,...
428425,ZZZ3,Gene_CellularComponent,GO:0072686,Gene,located_in,CellularComponent,GPKG::Monarch_KG,Generalised,NCBI_ID,Quick_GO,zinc finger ZZ-type containing 3,mitotic spindle
428426,ZZZ3,Gene_CellularComponent,GO:0140672,Gene,part_of,CellularComponent,Monarch_KG,Generalised,NCBI_ID,Quick_GO,zinc finger ZZ-type containing 3,ATAC complex
428427,ZZZ3,Gene_CellularComponent,GO:1902493,Gene,Hetionet::GpCC::Gene:Cellular Component,CellularComponent,DRKG::Harmonizome::Hetionet,Generalised,NCBI_ID,Quick_GO,zinc finger ZZ-type containing 3,acetyltransferase complex
428428,ZZZ3,Gene_CellularComponent,GO:1902494,Gene,None,CellularComponent,Harmonizome,Generalised,NCBI_ID,Quick_GO,zinc finger ZZ-type containing 3,catalytic complex


## 6 · Save Output

In [19]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 428,430 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_CELLULARCOMPONENT/ALL_GENE_CELLULARCOMPONENT.csv
